In [20]:
# IMPORTS

# Security for API tokens
from dotenv import dotenv_values
# spotify music
import spotipy
# spotify authorization
from spotipy.oauth2 import SpotifyOAuth

# Gemini API for Prompting
from google import genai



#import secrets
secrets = dotenv_values("../../.env.dev")

In [76]:
# Spotify LOGIN
scope_list = ['user-read-currently-playing','streaming', 'user-read-playback-position','user-read-recently-played','user-modify-playback-state', 'user-read-playback-state']
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=secrets['SPOTIFY_CLIENT_ID'], client_secret=secrets['SPOTIFY_CLIENT_SECRET'], redirect_uri=secrets['SPOTIFY_REDIRECT_URI'],scope=scope_list))

In [ ]:
# get current track listening to 
current_track = sp.current_user_playing_track() 
if current_track is not None and current_track.get('is_playing'):
    current_track_name = current_track['item']['name']
    current_track_artist = current_track['item']['artists'][0]['name']
    duration_ms = current_track['item']['duration_ms']
    progress_ms = current_track['progress_ms']
    time_left_ms = duration_ms - progress_ms
    time_left = int(time_left_ms / 1000)

    print(f"Currently playing: {current_track_name} by {current_track_artist}")
    print(f"Time Left: {time_left} seconds")



Currently playing: Te Irá Mejor Sin Mí by Joan Sebastian
Time Left: 145 seconds


['Te Irá Mejor Sin Mí by Joan Sebastian',
 'Te Irá Mejor Sin Mí by Joan Sebastian']

In [82]:
recently_played = []
list = sp.current_user_recently_played(limit=3)
if list is not None:
    for l in range(len(list['items'])):
        recently_played.append(list['items'][l]['track']['name'])

recently_played


['Ilusión De Amor', 'La Romana', 'Canalla']

In [ ]:
google_key = secrets["GEMINI_API_KEY"]

prompt = f'Find a song similar to {current_track_name} and not from this list: {recently_played} \n Reply with only the song name and artist of the song in this format: Title: <insert title>\n Artist: <insert artist name>'
client = genai.Client(api_key=google_key)
interaction = client.interactions.create(
    model = 'gemini-3.5-flash',
    input = prompt
)
output = interaction.output_text

In [65]:
new_song_artist = output.split(':')[-1]
new_song_title = output.split(':')[1].split('Artist')[0]
print("Title: " + new_song_title+"\nArtisit: " + new_song_artist)

Title:  Spike Spiegel

Artisit:  Saib


In [66]:
def get_song_uri(song_title, song_artist):
    query = song_title + " " + song_artist
    result = sp.search(q=query, limit=1, type="track")
    tracks = result.get('tracks', {}).get('items', [])
    if tracks:
        song_uri = tracks[0]['uri']
        return song_uri


upcoming_song_uri = get_song_uri(new_song_title,new_song_artist)

In [67]:
sp.add_to_queue(uri=upcoming_song_uri)